# full_3 — Channel-flow (A1) training via `train_channel.py`

Load the data into RAM once, then reuse it for each architecture trial.

**Set to 3 epochs for a smoke test.** Bump `EPOCHS` below when you're ready
for a full run.

In [1]:
import sys
from pathlib import Path

# make sure we can import train_channel.py from the same directory
_here = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

from train_channel import (
    load_training_data,
    run_trial,
    EPOCHS,
    BATCH_SIZE,
    NUM_WORKERS,
    S3_MODEL_PREFIX,
    S3_BUCKET,
    STORE_PATH,
)
print("train_channel imported OK")
print(f"  default EPOCHS   = {EPOCHS}")
print(f"  BATCH_SIZE       = {BATCH_SIZE}")
print(f"  NUM_WORKERS      = {NUM_WORKERS}")
print(f"  S3 dataset       = s3://{S3_BUCKET}/{STORE_PATH}")
print(f"  S3 model prefix  = {S3_MODEL_PREFIX}")

train_channel imported OK
  default EPOCHS   = 20
  BATCH_SIZE       = 4
  NUM_WORKERS      = 16
  S3 dataset       = s3://simulations/flow_export/A1
  S3 model prefix  = models/channel-1


## Config

In [2]:
# --- run config -------------------------------------------------------
EPOCHS_RUN = 20                       # smoke test; bump for a full run
LR_INIT    = 1e-3                    # initial LR (adaptive via ReduceLROnPlateau)
PROJECT    = "channel_full3_ptoq"         # Trackio project name
LABEL      = "full3"           # trial label for Trackio

# --- architecture knobs (sweepable) -----------------------------------
# DataNet from full_2.ipynb. Target depth and channel count come from the loaded data.
# c         : channel width of the 3D conv core
# n_blocks  : number of ConvBlock3D(2xConv3d+SiLU) blocks in the core
ARCH = {
    "c": 64,
    "n_blocks": 3,
}

print(f"config: epochs={EPOCHS_RUN}  lr={LR_INIT}  arch={ARCH}")

config: epochs=20  lr=0.001  arch={'c': 64, 'n_blocks': 3}


## Load data once

Rerun the load cell when changing the task, wall, sample limit, or spatial grid, or after restarting the kernel.

In [3]:
import os; print(os.getpid())

2619918


In [4]:
MAX_SAMPLES = 5000  # Per split; use None to load all train and validation samples.
TASK = 'wall_pressure_to_velocity'  # full_q_to_velocity / 'wall_pressure_to_q' / 'wall_pressure_to_velocity'.
WALL = None  # Wall tasks: None defaults to 'lower'; use 'upper' for the other wall.
HALF_Y = False  # True keeps y indices 0–64 inclusive on A1.
XZ_STRIDE = 1   # Set to 2 to use x[::2] and z[::2].
training_data = load_training_data(
    max_samples=MAX_SAMPLES, task=TASK, wall=WALL,
    half_y=HALF_Y, xz_stride=XZ_STRIDE, log=True,
)

  task=wall_pressure_to_velocity  wall=lower  half_y=False  xz_stride=1  T=4001  max_samples=5000
  train: [60:3196:1] (3136/3136 snapshots; loading)
  val: [3205:3597:1] (392/392 snapshots; loading)
  test: [3607:3999:1] (392/392 snapshots; held out, not loaded)


Load Zarr:   0%|          | 0/196 [00:00<?, ?chunk/s]

  [load] 3136 snapshots  x=0.19 GiB  y=74.07 GiB  peak RSS=149.46 GiB


Load Zarr:   0%|          | 0/25 [00:00<?, ?chunk/s]

  [load] 392 snapshots  x=0.02 GiB  y=9.26 GiB  peak RSS=149.46 GiB


## Run the trial

Change `ARCH` in the config cell, then rerun this cell without reloading the data.

In [ ]:
result = run_trial(
    ARCH,
    data=training_data,
    epochs=EPOCHS_RUN,
    lr=LR_INIT,
    trackio_project=PROJECT,
    trial_label=LABEL,
    log=True,
)

[trial] {'arch': {'c': 64, 'n_blocks': 3, 'dy': 129, 'out_ch': 3}, 'task': 'wall_pressure_to_velocity', 'wall': 'lower', 'half_y': False, 'xz_stride': 1, 'epochs': 20, 'lr': 0.001}
* Trackio project initialized: channel_full3_ptoq
* Trackio metrics logged to: /home/chancrin/.cache/huggingface/trackio
* View dashboard by running in your terminal:
trackio show --project "channel_full3_ptoq"
* or by running in Python: trackio.show(project="channel_full3_ptoq")
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: dainty-sunset-0
  model: 0.67M params


Train:   0%|          | 0/784 [00:00<?, ?batch/s]

## Result

In [ ]:
import json

# strip the live model out of the printable summary
summary = {k: v for k, v in result.items() if k != "model"}
print(json.dumps(summary, indent=2))
print()
print(f"best val MSE : {result['val_mse']:.6f}")
print(f"best epoch   : {result['best_epoch']}")
print(f"n params     : {result['n_params']:,}")
print(f"checkpoint   : {result['checkpoint_uri']}")